## Messaging

This example demonstrates the Messenger functionality in MMM-Audio, and the many kinds of messages that can be sent from Python to Mojo to control parameters in the audio graph.

We are able to send:
- Boolean values - .send_bool()
- Float values - .send_float()
- Lists of floats - .send_floats()
- Integer values - .send_int()
- Lists of integers - .send_ints()
- String values - .send_string() 
- Lists of strings - .send_strings()
- Trigger messages - .send_trig()

In [ ]:
import os
# Move up one level to the parent directory
os.chdir('..') 
print(os.getcwd())

from mmm_python import *

mmm_audio_instance = MMMAudio(2048, graph_name="MessengerExample", package_name="examples")

mmm_audio_instance.start_audio()

## In Mojo

To use the ```Messenger``` struct in Mojo, an instance of it is stored in a struct property of the target graph. 

```mojo

struct MessengerExample(Copyable, Movable):
    var world: World
    var m: Messenger # <---- The property to store the Messenger instance in
    var bool: Bool

    def __init__(out self, world: World):
        self.world = world
        self.m = Messenger(self.world) # The Messenger instance being created
        self.bool = False
```

Then, messages can then be handled in the the graph's ```next()``` method using the following ```Messenger``` methods:

* ```.update(address, variable)``` - Updates the value stored in the given variable when a new message is received on the given address.
* ```.notify_update(address, variable)``` - Updates the value stored in the given variable when a new message is received on the given address and returns True.
* ```.notify_trig(address)``` - Returns True when a new trigger message is received on the address.
* ```update_callback[call_back](address)``` - Executes a given callback function when a list is received.
* ```notify_callback[call_back](address)``` - Executes a given callback function when a list is received and returns True.

```mojo
def next(mut self) -> MFloat[2]:

        # This updates the value stored in the self.bool property and then returns True, causing the if statement to execute.
        if self.m.notify_update("bool", self.bool) :
            print("Bool value is now: " + String(self.bool))

        # This returns True, causing the if statement to execute.
        if self.m.notify_trig("trig"):
            print("Received trig")

        # A callback function
        def recv_floats(vals: List[Float64]) capturing -> None:
            print("Received the following floats: ", vals)

        # Executes the recv_floats() function when a list of floats is received at "callback".
        self.m.update_callback[recv_floats]("callback")

        
```



## In Python

It's possible to send boolean, integer, floats, strings, and triggers as well as lists of integers, floats, or strings. This is done by calling one of the following methods on an MMMAudio instance:

Single pieces of data
* ```.send_bool(adress, value)```
* ```.send_int(address, value)```
* ```.send_float(address, value)```
* ```.send_string(address, value)```
* ```.send_trig(address)```

Lists of data
* ```.send_ints(address, [values])```
* ```.send_floats(address, [values])```
* ```.send_strings(address, [values])```

Try sending some messages to Mojo below:

In [ ]:
mmm_audio_instance.send_bool("bool",True)  

In [ ]:
mmm_audio_instance.send_float("float", 440.0)

In [ ]:
mmm_audio_instance.send_floats("floats", [440.0, 550.0, 660.0])

In [ ]:
mmm_audio_instance.send_int("int", 42)

In [ ]:
mmm_audio_instance.send_ints("ints", [1, 22, 3, 4, 5])

In [ ]:
mmm_audio_instance.send_string("string", "Hello, World!")

In [ ]:
mmm_audio_instance.send_strings("strings", ["hello", "there", "general", "kenobi"])

In [ ]:
mmm_audio_instance.send_trig("trig")


## Things to know
New messages are processed at the beginning of every block, this causes only the last received message on an address each block to be to be executed. Any previous messages will be ignored. Additionally, larger block sizes will introduce some latency in the receipt of messages.

In [ ]:
mmm_audio_instance.send_bool("bool", True)
mmm_audio_instance.send_bool("bool", False) #Only this message is used

In [ ]:
mmm_audio_instance.send_bool("bool", False)
mmm_audio_instance.send_bool("bool", True) #Only this message is used

This can also cause problems when scheduling messages. Try to run the following code a few times. You'll notice that some times both ```.set_bool()``` messages are received, and other times only one is received (note that the block size for this mmm_audio_instace is set to 2048).

In [ ]:
import asyncio
import random

async def sched_event(delay, val):
    
    await asyncio.sleep(delay)
    mmm_audio_instance.send_bool("bool", val)
    
   

sched = Scheduler()

for b in [True, False]:
    sched.sched(sched_event(random.uniform(0.01, 0.1), b))

There are a two built in ways to address this. 

### Targeting Specific Messengers: Namespaces

If you need to control multiple synths using the same address ("freq" for example), you can target them by specifying a *namespace* when creating a ```Messenger```. This can be seen in the  MessengerExamples.mojo file, where the Tone struct has its own ```Messenger``` with each being given its own unique *namespace*:

```mojo
struct Tone(Movable,Copyable):
    var world: World
    var m: Messenger
    var freq: Float64
    var gate: Bool

    def __init__(out self, world: World, namespace: String):
        self.world = world
        self.m = Messenger(self.world,namespace)
        self.freq = 440.0
        self.gate = False

    def next(mut self) -> Float64:

        if self.m.notify_update("freq", self.freq) :
            print("Tone freq updated to ", self.freq)

        if self.m.notify_update("gate", self.gate) :
            print("Tone gate updated to ", self.gate)
```

```mojo
struct MessengerExample(Copyable, Movable):
    var world: World
    var m: Messenger
    var tones: List[Tone]

    def __init__(out self, world: World):
        self.world = world
        self.m = Messenger(self.world)

        self.tones = List[Tone]()
        # The for loop below assigns a unique namespace (tone_0, and tone_1) for each instance of Tone
        for i in range(2):
            self.tones.append(Tone(self.world, "tone_" + String(i)))

```

The namespace can then be targetted by the .send_x() methods by separating the namespace and address with a period (.)

In [ ]:

# Starts the synths
mmm_audio_instance.send_bool("tone_0.gate",True)
mmm_audio_instance.send_bool("tone_1.gate",True)

In [ ]:
# Sends a message to each Tone's "freq" address
mmm_audio_instance.send_float("tone_0.freq",440 * 1.059)
mmm_audio_instance.send_float("tone_1.freq",midicps(75))

In [ ]:
# Stops the synths
mmm_audio_instance.send_bool("tone_0.gate",False)
mmm_audio_instance.send_bool("tone_1.gate",False)

### Handling Messages with PolyPal

MMMAudio's ```Poly``` Mojo struct and ```PolyPal``` Python object can be used to send multiple messages at once to the same address. The ```Poly``` struct includes an internal Messenger that can be addressed through a *namespace*, which is specified with ```Poly```'s third argument.

```mojo
Poly(world: World, num_voices: Int, namespace: String)
```
Below is example code from the MidiSequencer.mojo example that shows the initialization of a ```Poly``` struct with the *namespace* "poly":

```mojo
struct MidiSequencer(Movable, Copyable):
    comptime num_messages = 10

    var world: World
    var current_voice: Int
    var messenger: Messenger
    var num_voices: Int

    var voices: List[TrigSynthVoice] #TrigSynthVoice is another struct in the MidiSequencer.mojo file.
    var poly: Poly

    def __init__(out self, world: World, num_voices: Int = 64):
        self.world = world
        self.num_voices = num_voices
        self.current_voice = 0

        self.messenger = Messenger(self.world)

        self.voices = [TrigSynthVoice(self.world) for _ in range(num_voices)]  # Initialize the list of voices

        self.poly = Poly(world, num_voices, "poly")

    @always_inline
    def next(mut self) -> MFloat[2]:
        var out = 0.0

        # the callback function sent to the Poly, to be called whenever a new trigger is received from Python.
        def call_back(mut voice: TrigSynthVoice, mut vals: List[Float64]) capturing -> None:
            voice.note = [vals[0], vals[1]]
        # the poly has an internal Messenger that receives messages from Python. these have to be in the form of a List[Float64] or a List[Int]. the callback function receives the list of ints or floats as the second argument, so the PolyObject can be controlled by the message from Python.
        self.poly.next_mtrig[call_back=call_back](self.voices)

        
```

The messages sent to ```Poly``` from Mojo are then handled in the graph's ```.next()``` method. Note that ```Poly```'s ```.next()``` method takes a callback function to handle the incoming data similar to ```.update_callback()```.

```mojo
def next(mut self) -> MFloat[2]:
    # 

     # the callback function sent to the Poly, to be called whenever a new trigger is received from Python.
    # the kinds of messages the Messenger can receive are defined by the type of the `note` argument in the callback function
    def callback(mut poly_object: OscVoice, mut vals: List[Int]) capturing -> None:
        if vals[1] > 0: # the call_back will be called for both note on and note off messages
            var midi = Float64(vals[0]) + poly_object.just_offset[vals[0] % 12]
            print(vals[0], midi)
            poly_object.freq = midicps(midi)
            poly_object.vol = Float64(vals[1]) / 127.0
    # the poly has an internal Messenger that receives messages from Python. these have to be in the form of a List[Float64] or a List[Int]
    # for next_mgate, the first value in the list is the note to trigger and the second value is the velocity or volume of the note, where 0 denotes a note off message. the callback function receives the list of ints or floats as the second argument, so the PolyObject can be controlled by the message from Python.
    self.poly.next_mgate[call_back=callback](self.voices)
```

In [1]:
# Load the MidiSequencer example file. Restart your notebook to stop the previous example.
import os
# Move up one level to the parent directory
os.chdir('..') 
print(os.getcwd())

from mmm_python import *

mmm_audio_instance = MMMAudio(2048, graph_name="MidiSequencer", package_name="examples")

mmm_audio_instance.start_audio()

/home/drewm/MMMAudio
[Main] Audio process started (PID: 80264)
[Child] Process started
[Child] Python: 3.14.4 (main, Aug 20 2026, 10:41:58) [GCC 15.2.0]
[PID 80264] Audio process starting...
[Compile] Starting compile for MidiSequencer from examples
Compiled Mojo graph 'MidiSequencer' from package 'examples'. It is ready to run.


ALSA lib confmisc.c:855:(parse_card) [error.core] cannot find card '0'
ALSA lib conf.c:5211:(_snd_config_evaluate) [error.core] function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) [error.core] error evaluating strings
ALSA lib conf.c:5211:(_snd_config_evaluate) [error.core] function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) [error.core] error evaluating name
ALSA lib conf.c:5211:(_snd_config_evaluate) [error.core] function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5734:(snd_config_expand) [error.core] Evaluate error: No such file or directory
ALSA lib pcm.c:2722:(snd_pcm_open_noupdate) [error.pcm] Unknown PCM sysdefault
ALSA lib confmisc.c:855:(parse_card) [error.core] cannot find card '0'
ALSA lib conf.c:5211:(_snd_config_evaluate) [error.core] function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(

MMMWorld initialized with sample rate: 44100.0
Environment pointer is set.
[PID 80264] Using input device: default
[PID 80264] Using output device: default
[PID 80264] Sample rate: 44100, Block size: 2048
[PID 80264] Input channels: 2, Output channels: 2
[Main] Audio process ready, sample rate: 44100[PID 80264] Audio process ready

Xlib.xauth: warning, no xauthority details available
Xlib.xauth: warning, no xauthority details available
[PID 80264] Audio activated


On the Python side, MMMAudio's ```PolyPal``` allows for easy sending of messages that ```Poly``` understands. ```PolyPal``` only allows for the sending of floats and ints, either individually or in homogenous lists with the following methods:

* ```PolyPal.send_int(value)``` - Send a single integer to the ```Poly``` struct at the target *namespace*
* ```PolyPal.send_ints([values])``` - Send a list of integers to the ```Poly``` struct at the target *namespace*
* ```PolyPal.send_float(value)``` - Send a single float to the ```Poly``` struct at the target *namespace*
* ```PolyPal.send_floats([values])``` - Send a list of floats to the ```Poly``` struct at the target *namespace*

Execute the code blocks below to test the ```PolyPal```

In [2]:
new_poly_pal = PolyPal(mmm_audio_instance, "poly", 16)

Send a single message to ```Poly```

In [19]:
new_poly_pal.send_floats([72, 1.0])

Send three messages simultaneously to ```Poly```

In [23]:
import random
new_poly_pal.send_floats([random.randint(48, 62), 1.0])
new_poly_pal.send_floats([random.randint(58, 72), 1.0])
new_poly_pal.send_floats([random.randint(68, 82), 1.0])

Send 16 messages simultaneously to ```Poly``` (all alvailable voices)

In [ ]:
[new_poly_pal.send_floats([random.uniform(48, 127), random.uniform(0.1, 0.5)]) for _ in range(16)]